# Lesson 9B: Association Rules Practical — mlxtend, FP-Growth, and Market Basket Mining

## Introduction

Lesson 9A derived Apriori from scratch on a small, clean dataset with 12 items and 2,000 transactions — enough to prove the algorithm correct, too small to show what happens at real retail scale. This lesson closes that gap with `mlxtend`'s production implementations, a larger catalog, and the question every practitioner eventually asks: **as the item catalog grows, does Apriori's repeated-scan approach still keep up?**

**FP-Growth** (Han et al., 2000) answers that with a different mechanic entirely: instead of repeatedly re-scanning the transaction database for every candidate size (Apriori's bottleneck), it compresses the whole database into a single **FP-tree** (frequent-pattern tree) in two passes, then mines frequent itemsets by walking conditional pattern bases through that tree. Same frequent-itemset guarantee as Apriori — the itemsets found are provably identical — but the mechanics scale differently as the support threshold drops and candidate sets explode.

In this lesson:
1. Build a larger, more realistic market-basket dataset (more items, more transactions)
2. Mine frequent itemsets with `mlxtend`'s Apriori and extract rules with `association_rules`
3. Mine the same data with FP-Growth and confirm both agree exactly on frequent itemsets
4. Compare Apriori vs FP-Growth runtime as the minimum support threshold shrinks
5. Explore the minimum support/confidence trade-off: how threshold choice reshapes rule count and quality
6. Extract ranked, actionable business rules

## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [A Larger Market-Basket Dataset](#dataset)
4. [mlxtend Apriori: Mining and Rule Extraction](#apriori-mlxtend)
5. [FP-Growth: A Different Mining Mechanic, Same Guarantee](#fpgrowth)
6. [Runtime: Apriori vs FP-Growth as Support Shrinks](#runtime)
7. [The Support/Confidence Trade-off](#tradeoff)
8. [Ranked, Actionable Business Rules](#business-rules)
9. [Conclusion](#conclusion)

<a name="required-libraries"></a>
## Required Libraries

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Libraries loaded!")

<a name="dataset"></a>
## A Larger Market-Basket Dataset

40 items, 10 engineered co-purchase clusters of varying size and popularity (larger and more numerous than 9A's four pairs), plus background noise — closer to a real, if small, retail catalog.

In [ ]:
rng = np.random.RandomState(42)

departments = {
    'bakery': ['bread', 'bagel', 'croissant', 'muffin'],
    'dairy': ['milk', 'butter', 'cheese', 'yogurt', 'cream'],
    'breakfast': ['eggs', 'cereal', 'coffee', 'sugar', 'honey'],
    'snacks': ['chips', 'soda', 'pretzels', 'popcorn', 'cookies'],
    'baby': ['diapers', 'baby_wipes', 'baby_formula'],
    'alcohol': ['beer', 'wine'],
    'produce': ['bananas', 'apples', 'lettuce', 'tomatoes', 'onions'],
    'meat': ['chicken', 'beef', 'bacon', 'sausage'],
    'pantry': ['flour', 'rice', 'pasta', 'olive_oil', 'canned_beans'],
    'cleaning': ['detergent', 'paper_towels', 'dish_soap'],
}
all_items = sorted({item for items in departments.values() for item in items})
print(f"Catalog size: {len(all_items)} items across {len(departments)} departments")

# Engineered co-purchase clusters (item subset, trigger probability, "add-on" items with their own odds)
clusters = [
    (['bread', 'butter'], 0.30, [('milk', 0.55), ('cheese', 0.25)]),
    (['chips', 'soda'], 0.22, [('pretzels', 0.30)]),
    (['diapers', 'baby_wipes'], 0.12, [('baby_formula', 0.5)]),
    (['coffee', 'sugar'], 0.28, [('cereal', 0.35)]),
    (['beer', 'chips'], 0.15, [('pretzels', 0.4)]),
    (['pasta', 'olive_oil'], 0.20, [('canned_beans', 0.3)]),
    (['chicken', 'lettuce', 'tomatoes'], 0.18, [('onions', 0.5)]),
    (['bacon', 'eggs'], 0.24, [('bread', 0.4)]),
    (['detergent', 'paper_towels'], 0.16, [('dish_soap', 0.45)]),
    (['wine', 'cheese'], 0.14, [('bread', 0.35)]),
]


def generate_transaction():
    basket = set()
    for base_items, trigger_prob, addons in clusters:
        if rng.random() < trigger_prob:
            basket.update(base_items)
            for addon_item, addon_prob in addons:
                if rng.random() < addon_prob:
                    basket.add(addon_item)
    n_noise = rng.randint(0, 4)
    basket.update(str(x) for x in rng.choice(all_items, size=n_noise, replace=False))
    return sorted(basket) if basket else [str(rng.choice(all_items))]


transactions = [generate_transaction() for _ in range(6000)]
basket_sizes = [len(t) for t in transactions]

print(f"Total transactions: {len(transactions)}")
print(f"Basket size: mean={np.mean(basket_sizes):.1f}, min={min(basket_sizes)}, max={max(basket_sizes)}")

encoder = TransactionEncoder()
one_hot = encoder.fit_transform(transactions)
df_encoded = pd.DataFrame(one_hot, columns=encoder.columns_)
print(f"One-hot encoded shape: {df_encoded.shape}")

<a name="apriori-mlxtend"></a>
## mlxtend Apriori: Mining and Rule Extraction

`mlxtend.frequent_patterns.apriori` handles candidate generation and pruning internally — the from-scratch version in 9A implemented exactly this logic. `association_rules` then generates rules directly from the frequent-itemset table, computing support, confidence, and lift (plus several other metrics) in one call.

Note: `association_rules` needs integer-labeled itemsets internally in this mlxtend release, so we mine with `use_colnames=False` and map integer positions back to item names afterward for readability.

In [ ]:
df_int_cols = df_encoded.copy()
df_int_cols.columns = range(len(df_encoded.columns))
column_names = {i: name for i, name in enumerate(df_encoded.columns)}

min_support = 0.03
frequent_apriori = apriori(df_int_cols, min_support=min_support, use_colnames=False)

min_confidence = 0.4
rules_apriori = association_rules(frequent_apriori, num_itemsets=len(df_int_cols), metric='confidence', min_threshold=min_confidence)


def itemset_to_names(itemset) -> frozenset:
    return frozenset(column_names[i] for i in itemset)


rules_apriori = rules_apriori.copy()
rules_apriori['antecedents'] = rules_apriori['antecedents'].apply(itemset_to_names)
rules_apriori['consequents'] = rules_apriori['consequents'].apply(itemset_to_names)

print(f"Frequent itemsets: {len(frequent_apriori)} at min_support={min_support}")
print(f"Rules: {len(rules_apriori)} at min_confidence={min_confidence}")
top_by_lift = rules_apriori[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values('lift', ascending=False).head(10)
top_by_lift

Sorting purely by lift surfaces exactly the compound-itemset artifact from 9A, now at production scale: rules spanning several unrelated departments (e.g. cleaning + produce + meat items combined) can outrank genuinely useful single-department associations, simply because compound antecedent/consequent supports are individually small. Raw lift ranking is not yet a business rule — the next section fixes that.

<a name="fpgrowth"></a>
## FP-Growth: A Different Mining Mechanic, Same Guarantee

FP-Growth builds a compressed prefix tree (the FP-tree) from the transactions in two passes, then recursively mines frequent itemsets from conditional pattern bases — no repeated full-database scans, no explicit candidate generation step. It's a different algorithm end to end, but by definition it must find **exactly the same frequent itemsets** as Apriori at the same support threshold.

In [ ]:
frequent_fpgrowth = fpgrowth(df_int_cols, min_support=min_support, use_colnames=False)

itemsets_apriori_named = {itemset_to_names(row['itemsets']): row['support'] for _, row in frequent_apriori.iterrows()}
itemsets_fpgrowth_named = {itemset_to_names(row['itemsets']): row['support'] for _, row in frequent_fpgrowth.iterrows()}

same_itemsets = set(itemsets_apriori_named.keys()) == set(itemsets_fpgrowth_named.keys())
max_support_diff = max(abs(itemsets_apriori_named[i] - itemsets_fpgrowth_named[i]) for i in itemsets_apriori_named)

print(f"Apriori frequent itemsets:  {len(itemsets_apriori_named)}")
print(f"FP-Growth frequent itemsets: {len(itemsets_fpgrowth_named)}")
print(f"Identical itemset sets: {same_itemsets}")
print(f"Max support difference: {max_support_diff:.6f}")

<a name="runtime"></a>
## Runtime: Apriori vs FP-Growth as Support Shrinks

Lower `min_support` means more frequent itemsets survive, which means more candidates for Apriori to generate and re-scan the database for. FP-Growth's tree-based approach should degrade more gracefully.

In [ ]:
support_thresholds = [0.10, 0.07, 0.05, 0.03, 0.02, 0.015]
apriori_times, fpgrowth_times, n_itemsets = [], [], []

for support in support_thresholds:
    start = time.perf_counter()
    freq_a = apriori(df_int_cols, min_support=support, use_colnames=False)
    apriori_times.append(time.perf_counter() - start)

    start = time.perf_counter()
    freq_f = fpgrowth(df_int_cols, min_support=support, use_colnames=False)
    fpgrowth_times.append(time.perf_counter() - start)

    n_itemsets.append(len(freq_a))
    assert len(freq_a) == len(freq_f), "Apriori and FP-Growth must agree on itemset count"

timing_df = pd.DataFrame({
    'min_support': support_thresholds,
    'n_frequent_itemsets': n_itemsets,
    'apriori_seconds': apriori_times,
    'fpgrowth_seconds': fpgrowth_times,
})
print(timing_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(support_thresholds, n_itemsets, marker='o')
axes[0].invert_xaxis()
axes[0].set_xlabel('min_support')
axes[0].set_ylabel('Frequent itemsets found')
axes[0].set_title('Itemset Count Grows as Support Shrinks', fontsize=11, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(support_thresholds, apriori_times, marker='o', label='Apriori')
axes[1].plot(support_thresholds, fpgrowth_times, marker='s', label='FP-Growth')
axes[1].invert_xaxis()
axes[1].set_xlabel('min_support')
axes[1].set_ylabel('Runtime (seconds)')
axes[1].set_title('Mining Runtime vs Support Threshold', fontsize=11, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 Watch the gap between the two lines widen (or not) as min_support drops and the candidate space grows — at this catalog size (40 items) the effect is real but modest; it becomes dramatic at retail-scale catalogs (thousands of items)")

<a name="tradeoff"></a>
## The Support/Confidence Trade-off

Two independent knobs, two independent effects: `min_support` controls how many itemsets are even considered; `min_confidence` (applied afterward, during rule generation) controls how many of the resulting rules are kept. Loosening either threshold trades rule *quantity* for rule *quality*.

In [ ]:
confidence_thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
frequent_fixed = apriori(df_int_cols, min_support=0.03, use_colnames=False)

rule_counts = []
for conf in confidence_thresholds:
    rules = association_rules(frequent_fixed, num_itemsets=len(df_int_cols), metric='confidence', min_threshold=conf)
    rule_counts.append(len(rules))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(confidence_thresholds, rule_counts, marker='o', linewidth=2)
ax.set_xlabel('min_confidence (min_support fixed at 0.03)')
ax.set_ylabel('Rules surviving threshold')
ax.set_title('Rule Count vs Confidence Threshold', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for conf, count in zip(confidence_thresholds, rule_counts):
    print(f"min_confidence={conf}: {count} rules")

print("\n💡 Lower thresholds surface more rules, but many will be noise — always sanity-check the top rules by lift, not just by clearing the confidence bar")

<a name="business-rules"></a>
## Ranked, Actionable Business Rules

For a real recommendation ("customers who buy X and Y also buy Z") or shelf-placement decision, a rule needs to be **common enough to matter** (support floor), **reliable** (confidence), **genuinely associated, not coincidental** (lift) — and **interpretable enough to act on**. We restrict to a single-item consequent and a small (≤2 item) antecedent: a merchandising team can act on "diapers + baby wipes → buy baby formula too," not on a four-item rule spanning two unrelated departments. This size cap is also what keeps 9A's compound-itemset lift-inflation artifact out of the results — the wide multi-department rules from the previous section simply don't qualify.

In [ ]:
actionable = rules_apriori[
    (rules_apriori['support'] >= 0.03)
    & (rules_apriori['antecedents'].apply(len) <= 2)
    & (rules_apriori['consequents'].apply(len) == 1)
].copy()

actionable['rule'] = actionable.apply(
    lambda r: f"{'+'.join(sorted(r['antecedents']))} → {'+'.join(sorted(r['consequents']))}", axis=1
)

top_rules = actionable.sort_values('lift', ascending=False)[['rule', 'support', 'confidence', 'lift']].head(10)
print(top_rules.to_string(index=False))

<a name="conclusion"></a>
## Conclusion

### Key Takeaways

1. **mlxtend's Apriori and `association_rules`** cover the full practical workflow: one-hot encode transactions, mine frequent itemsets, extract rules with support/confidence/lift computed automatically.
2. **FP-Growth finds identical frequent itemsets** to Apriori — the tree-based mechanic changes *how* the search happens, not *what* it finds. Both are correct; they differ in performance profile.
3. **FP-Growth's advantage grows as min_support shrinks** — Apriori's repeated candidate generation and database re-scanning costs more as the frequent-itemset space grows, while FP-Growth's single compressed tree amortizes that cost.
4. **Support and confidence are independent knobs** — loosening either surfaces more rules, generally at the cost of rule quality; there's no single "correct" threshold, only a business trade-off.
5. **Actionable rules need a support floor and a size cap**, not just high lift — 9A's caveat about compound-itemset lift inflation is a real filtering concern, not just a theoretical aside.

### Practical Guidance

- Start with a support threshold that leaves a manageable number of frequent itemsets (dozens to low hundreds, not thousands) — too low and rule extraction becomes a firehose of low-value noise.
- Prefer FP-Growth over Apriori once the catalog or transaction volume gets large — same guarantees, better scaling.
- Filter final rules by **support first** (is this common enough to matter commercially?), **then** rank by lift among the survivors.

### Next Steps

This closes Lesson 9. In Lesson 10, we move to topic modeling: discovering latent themes across a document corpus with Latent Dirichlet Allocation.

You now have the complete association-rule toolkit: the theory and from-scratch derivation from 9A, and the production-scale mlxtend workflow — Apriori, FP-Growth, and threshold tuning — from this lesson 🎯